# Terror Infinity RPG — Enemy Portraits (The Hive)

Generates one dramatic portrait per enemy using **Animagine XL 3.1** (SDXL).

**Enemy roster** (Resident Evil universe, The Hive + Raccoon City arc):

| Slug | Enemy | First appears |
|---|---|---|
| `zombie` | Standard Hive zombie (Umbrella worker) | RE Movie 1 |
| `zombie-dog` | Zombie Doberman / Cerberus | RE Movie 1 |
| `licker` | Licker (main creature boss) | RE Movie 1 |
| `zombie-commando` | Armed Umbrella security zombie | RE Movie 1 |
| `red-queen` | The Red Queen (AI hologram) | RE Movie 1 |
| `crimson-head` | Crimson Head (enhanced fast zombie) | RE Game / Movie |
| `t103-tyrant` | T-103 Tyrant | RE Movie 2 |
| `nemesis` | Nemesis | RE Apocalypse |
| `licker-beta` | Licker Beta (evolved) | RE Afterlife |
| `hunter-alpha` | Hunter Alpha | RE Game / RE3 |

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"\nCUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q -U diffusers transformers accelerate safetensors

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓  Output → MyDrive/terror-infinity-enemies/")

In [ ]:
# ── Cell 4: Load Animagine XL 3.1 ────────────────────────────────────────────
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

print("Loading Animagine XL 3.1 ...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    "cagliostrolab/animagine-xl-3.1",
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True,
)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Pipeline ready ✓  VRAM: {used:.1f} / {total:.1f} GB")

In [ ]:
# ── Cell 5: Enemy definitions ─────────────────────────────────────────────────
#
# Each entry:
#   slug     → output filename: {slug}.png
#   prompt   → full positive prompt (prefix already included)
#   negative → custom negative for this enemy type
#
# Style notes:
#   - 2.5D anime horror = clean lineart with strong horror mood (not photorealistic)
#   - 'full color' + explicit color anchors prevent grayscale renders
#   - Anti-duplicate tags go FIRST in negative (before quality terms) so they
#     are never truncated at the 77-token CLIP limit
#

_QH = ('masterpiece, best quality, very aesthetic, '
       '2.5D, anime style, horror, full color, '
       'dark atmosphere, dramatic lighting, ')

NEG_BASE = (
    'duplicate, clone, multiple creatures, double exposure, '
    'nsfw, lowres, (bad), text, error, worst quality, low quality, jpeg artifacts, '
    'watermark, blurry, deformed, '
    'monochrome, grayscale, black and white, '
    'cute, happy, cheerful, bright sunny colors'
)
NEG_MONSTER  = NEG_BASE + ', human face, human body, anime girl, anime boy'
NEG_ZOMBIE   = NEG_BASE + ', healthy, alive, clean clothes'
NEG_HOLOGRAM = NEG_BASE + ', flesh, real person, photorealistic skin'

ENEMIES = [
    {
        'slug':     'zombie',
        'label':    'Hive Zombie',
        'prompt':   _QH + (
            'half body, solo, zombie, Umbrella worker, '
            'grey rotting skin, dead white eyes, torn blue uniform, '
            'open mouth, blood stains, dark corridor background, green mist'
        ),
        'negative': NEG_ZOMBIE,
    },
    {
        'slug':     'zombie-dog',
        'label':    'Zombie Dog / Cerberus',
        'prompt':   _QH + (
            'full body, solo, zombie doberman, '
            'rotting black fur, exposed ribs, red glowing eyes, '
            'snarling attack stance, bloody drool, dark background'
        ),
        'negative': NEG_MONSTER,
    },
    {
        'slug':     'licker',
        'label':    'Licker',
        'prompt':   _QH + (
            'solo, Licker creature, no skin, '
            'exposed wet red muscle tissue, brain on skull, '
            'long tongue, bone claws, wall ceiling crawl, '
            'dark laboratory background, blood walls'
        ),
        'negative': NEG_MONSTER + ', human, person, dressed',
    },
    {
        'slug':     'zombie-commando',
        'label':    'Zombie Commando',
        'prompt':   _QH + (
            'half body, solo, zombie soldier, '
            'torn Umbrella tactical gear, dead pale skin, '
            'bloodshot dead eyes, assault rifle, '
            'dark corridor, red emergency light'
        ),
        'negative': NEG_ZOMBIE,
    },
    {
        'slug':     'red-queen',
        'label':    'The Red Queen',
        'prompt':   _QH + (
            'holographic girl, transparent red glow, '
            'floating midair, empty evil smile, short neat hair, '
            'large empty eyes, server room background, '
            'red light particles, digital artifact, eerie'
        ),
        'negative': NEG_HOLOGRAM + ', zombie, monster, blood, gore',
    },
    {
        'slug':     'crimson-head',
        'label':    'Crimson Head',
        'prompt':   _QH + (
            'half body, solo, crimson head zombie, '
            'dark red cracked skin, sunken face, bared teeth, '
            'claw hands, bloodshot eyes, speed lines, '
            'dark underground background'
        ),
        'negative': NEG_ZOMBIE,
    },
    {
        'slug':     't103-tyrant',
        'label':    'T-103 Tyrant',
        'prompt':   _QH + (
            'upper body, solo, massive humanoid monster, '
            'pale grey skin, exposed pulsing heart on chest, '
            'black tattered trench coat, enormous bone claw arm, '
            'dead pale eyes, dark rainy background, cold mist'
        ),
        'negative': NEG_MONSTER,
    },
    {
        'slug':     'nemesis',
        'label':    'Nemesis',
        'prompt':   _QH + (
            'upper body, solo, Nemesis monster, '
            'massive pale stitched skin, black long coat, '
            'tentacle arm, yellow visor eye, rocket launcher, '
            'Raccoon City night, rain, fire background'
        ),
        'negative': NEG_MONSTER,
    },
    {
        'slug':     'licker-beta',
        'label':    'Licker Beta (evolved)',
        'prompt':   _QH + (
            'solo, evolved Licker creature, '
            'larger, dark crimson muscle, multiple red eyes, '
            'extended bone claws, multi-tongue, '
            'ceiling attack pose, blood walls, dark background'
        ),
        'negative': NEG_MONSTER + ', human, person',
    },
    {
        'slug':     'hunter-alpha',
        'label':    'Hunter Alpha',
        'prompt':   _QH + (
            'upper body, solo, Hunter creature, '
            'bipedal reptile mutant, pale grey scales, '
            'exposed jaw bone, claw hands, red eyes, '
            'muscular, growling, dark forest night background'
        ),
        'negative': NEG_MONSTER + ', human face',
    },
]

print(f"Enemies    : {len(ENEMIES)}")
print(f"Total      : {len(ENEMIES)} images  (~{max(1, len(ENEMIES) * 30 // 60)} min on Colab T4)")
print()
for e in ENEMIES:
    print(f"  {e['slug']:<22}  {e['label']}")

# ── Token sanity check ───────────────────────────────────────────────────────
print()
try:
    from transformers import CLIPTokenizer
    tok = CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')
    print(f"  {'Slug':<22} {'Pos tokens':>10} {'Neg tokens':>10}  Status")
    print('  ' + '-' * 52)
    for e in ENEMIES:
        tp = len(tok(e['prompt']       )['input_ids'])
        tn = len(tok(e['negative']     )['input_ids'])
        flag = '⚠ OVER 77' if max(tp, tn) > 77 else '✓'
        print(f"  {e['slug']:<22} {tp:>10} {tn:>10}   {flag}")
except Exception as ex:
    print(f'(Token check skipped: {ex})')

In [ ]:
# ── Cell 6: Generate all enemy portraits ─────────────────────────────────────
import hashlib, os
from IPython.display import display, Image as IPImage
from PIL import Image as PILImage, ImageDraw, ImageFont

OUTPUT_DIR = '/content/enemy-portraits'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_W, IMG_H = 832, 1216
STEPS        = 30       # slightly more steps for complex creatures
CFG          = 7.5      # slightly higher CFG for horror detail

def stable_seed(text):
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16) % 999983

results = []

print('=' * 55)
print('  ENEMY PORTRAITS  (Resident Evil universe)')
print('=' * 55)

for idx, enemy in enumerate(ENEMIES):
    slug  = enemy['slug']
    label = enemy['label']
    seed  = stable_seed(slug)

    print(f"\n[{idx+1}/{len(ENEMIES)}]  {label}  (seed {seed})")
    print(f'  Generating ... ', end='', flush=True)

    gen = torch.Generator('cuda').manual_seed(seed)
    img = pipe(
        prompt          = enemy['prompt'],
        negative_prompt = enemy['negative'],
        width           = IMG_W,
        height          = IMG_H,
        num_inference_steps = STEPS,
        guidance_scale  = CFG,
        generator       = gen,
    ).images[0]

    path = f'{OUTPUT_DIR}/{slug}.png'
    img.save(path)
    results.append((slug, path))
    print('saved')

    # ── Preview with label ───────────────────────────────────────────────────
    display(IPImage(path, width=280))
    print(f'  ^ {label}')

print(f"\n✓  {len(results)} enemy portraits generated")

In [ ]:
# ── Cell 7: Save to Google Drive + download zip ───────────────────────────────
import shutil, os
from google.colab import files

DRIVE_OUTPUT = '/content/drive/MyDrive/terror-infinity-enemies'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

copied = 0
for slug, path in results:
    if os.path.exists(path):
        shutil.copy(path, f'{DRIVE_OUTPUT}/{slug}.png')
        copied += 1

print(f'Saved {copied} enemy portraits to Google Drive → terror-infinity-enemies/')

shutil.make_archive('/content/enemy-portraits', 'zip', OUTPUT_DIR)
print('Downloading enemy-portraits.zip ...')
files.download('/content/enemy-portraits.zip')